## Consolidação e Verificação da Qualidade dos Dados

No notebook anterior, utilizamos apenas os dados de 2025 para entender a estrutura do dataset. Agora, vamos trabalhar com todo o período disponível, entre 2000 e 2025.

Nesta etapa, vamos:

- juntar todos os arquivos anuais em um único DataFrame;
- verificar os tipos das colunas, valores ausentes, registros duplicados e possíveis inconsistências;
- organizar e salvar o arquivo consolidado na pasta data/processed.

### 1. Importação das Bibliotecas


In [34]:
from pathlib import Path
import pandas as pd

RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')

### 2. Concatenando os Dados e Formando o Dataset Histórico

Primeiro, vamos localizar todos os arquivos Parquet que estão na pasta data/raw. Depois, esses arquivos serão lidos e reunidos em um único DataFrame.

Com isso, teremos todo o histórico de carga elétrica entre 2000 e 2025 em uma única base.


In [35]:
files = sorted(RAW_PATH.glob('curva_carga_*.parquet'))

print(f'{len(files)} arquivos encontrados.')

electric_data = pd.concat([pd.read_parquet(f) for f in files], ignore_index = True)
electric_data.head()

26 arquivos encontrados.


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2000-01-01 00:00:00,2373.69999999
1,NE,NORDESTE,2000-01-01 00:00:00,5340.20000000
2,S,SUL,2000-01-01 00:00:00,5777.00000000
3,SE,SUDESTE,2000-01-01 00:00:00,21182.99999999
4,N,NORTE,2000-01-01 01:00:00,2331.60000000


#### 2.1. Verificando a Data Inicial e Final

Após a junção dos arquivos, vamos confirmar se o período começa em 2000 e termina em 2025, como esperado.


In [36]:
print(f"Data Inicial: {electric_data['din_instante'].min()}")
print(f"Data Final: {electric_data['din_instante'].max()}")

Data Inicial: 2000-01-01 00:00:00
Data Final: 2025-12-31 23:00:00


#### 2.2. Dimensão do Dataset

Agora, vamos verificar a quantidade de linhas e colunas do dataset consolidado.


In [37]:
print(f'O dataset final possui um total de {electric_data.shape[0]} linhas e {electric_data.shape[1]} colunas.')

O dataset final possui um total de 911608 linhas e 4 colunas.


### 3. Informações Gerais

Antes de continuar, vamos utilizar o método info para observar os tipos das colunas, a quantidade de valores preenchidos e o uso de memória do dataset.

A partir desse resultado, podemos identificar se existe alguma coluna que precisa ser analisada com mais atenção.


In [38]:
electric_data.info(memory_usage = 'deep')

<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911608 non-null  object        
dtypes: datetime64[ns](1), object(1), str(2)
memory usage: 80.4 MB


A primeira coisa que chamou atenção foi a coluna de carga elétrica. Ela apareceu com o tipo object, mas seus valores deveriam ser numéricos, pois representam cargas com casas decimais.

Antes de fazer qualquer conversão, vamos verificar como essa coluna está armazenada nos arquivos de cada ano.

#### 3.1. Tipo da Coluna de Carga em Cada Ano


In [39]:
for file in files:
    year_data = pd.read_parquet(file, columns = ['val_cargaenergiahomwmed'])
    year = file.stem.split('_')[-1]
    column_type = year_data['val_cargaenergiahomwmed'].dtype

    print(f'{year}: {column_type}')

2000: str
2001: str
2002: str
2003: str
2004: str
2005: str
2006: str
2007: str
2008: str
2009: str
2010: str
2011: str
2012: str
2013: str
2014: str
2015: str
2016: str
2017: str
2018: str
2019: str
2020: str
2021: str
2022: str
2023: str
2024: str
2025: float64


In [40]:
empty_load = electric_data[electric_data['val_cargaenergiahomwmed'] == '']

print(f'Quantidade de valores vazios: {empty_load.shape[0]}')

empty_load.head()

Quantidade de valores vazios: 259


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
487912,N,NORTE,2013-12-01 00:00:00,
487913,NE,NORDESTE,2013-12-01 00:00:00,
487914,S,SUL,2013-12-01 00:00:00,
487915,SE,SUDESTE,2013-12-01 00:00:00,
487916,N,NORTE,2013-12-01 01:00:00,


O resultado mostrou que a coluna de carga está como texto nos arquivos de 2000 a 2024. Apenas em 2025 ela já aparece como float64.

Como arquivos com tipos diferentes foram concatenados, o Pandas manteve a coluna final como object.

Também encontramos 259 registros preenchidos com uma string vazia. Esses campos parecem vazios para nós, mas ainda são considerados textos válidos pelo Pandas. Agora, vamos verificar o que acontece durante a conversão.

#### 3.2. Conversão da Coluna de Carga


In [41]:
after_conversion = pd.to_numeric(
    electric_data['val_cargaenergiahomwmed'],
    errors='coerce'
)

print(
    f"Quantidade de valores ausentes antes da conversão: "
    f"{electric_data['val_cargaenergiahomwmed'].isna().sum()}"
)

print(
    f"Quantidade de valores ausentes após a conversão: "
    f"{after_conversion.isna().sum()}"
)

Quantidade de valores ausentes antes da conversão: 0
Quantidade de valores ausentes após a conversão: 259


Na primeira verificação com o método info, a coluna não aparecia com valores nulos. Isso aconteceu porque os 259 campos vazios ainda estavam armazenados como texto.

Após a conversão, esses campos passaram a ser reconhecidos como NaN e a coluna pôde ser representada como float64.

Agora, vamos aplicar essa conversão no dataset e conferir novamente suas informações.


In [42]:
electric_data['val_cargaenergiahomwmed'] = after_conversion

In [43]:
electric_data.info(memory_usage = 'deep')

<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911349 non-null  float64       
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 34.1 MB


### 4. Verificação dos Valores Ausentes

Com o tipo da coluna corrigido, vamos contar os valores ausentes em cada variável.

Depois, criaremos um recorte apenas com as cargas nulas para verificar em quais anos, datas e subsistemas elas aparecem.


In [44]:
electric_data.isna().sum()

id_subsistema                0
nom_subsistema               0
din_instante                 0
val_cargaenergiahomwmed    259
dtype: int64

In [45]:
missing_data = electric_data[electric_data['val_cargaenergiahomwmed'].isna()].copy()
missing_data.head()

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
487912,N,NORTE,2013-12-01 00:00:00,NaN
487913,NE,NORDESTE,2013-12-01 00:00:00,NaN
487914,S,SUL,2013-12-01 00:00:00,NaN
487915,SE,SUDESTE,2013-12-01 00:00:00,NaN
487916,N,NORTE,2013-12-01 01:00:00,NaN


In [46]:
missing_data['ano'] = missing_data['din_instante'].dt.year
missing_data['ano'].value_counts().sort_index()

ano
2013    96
2014    76
2015    76
2016     4
2017     4
2018     3
Name: count, dtype: int64

Os valores ausentes aparecem entre 2013 e 2018. As maiores quantidades estão nos anos de 2013, 2014 e 2015.

Para entender o que aconteceu, vamos detalhar esses registros por data e subsistema.


In [47]:
missing_data['data'] = missing_data['din_instante'].dt.date

pd.crosstab(missing_data['data'], missing_data['id_subsistema'])

id_subsistema,N,NE,S,SE
data,,,,
2013-12-01,24,24,24,24
2014-02-01,0,24,24,24
2014-10-19,1,1,1,1
2015-04-09,0,24,24,24
2015-10-18,1,1,1,1
2016-10-16,1,1,1,1
2017-10-15,1,1,1,1
2018-11-04,1,1,0,1


In [48]:
missing_data.groupby(['data', 'id_subsistema'])['din_instante'].agg(['count', 'nunique', 'min', 'max'])

count  nunique        min                 max
data       id_subsistema                                               
2013-12-01 N                 24       24 2013-12-01 2013-12-01 23:00:00
           NE                24       24 2013-12-01 2013-12-01 23:00:00
           S                 24       24 2013-12-01 2013-12-01 23:00:00
           SE                24       24 2013-12-01 2013-12-01 23:00:00
2014-02-01 NE                24       24 2014-02-01 2014-02-01 23:00:00
           S                 24       24 2014-02-01 2014-02-01 23:00:00
           SE                24       24 2014-02-01 2014-02-01 23:00:00
2014-10-19 N                  1        1 2014-10-19 2014-10-19 00:00:00
           NE                 1        1 2014-10-19 2014-10-19 00:00:00
           S                  1        1 2014-10-19 2014-10-19 00:00:00
           SE                 1        1 2014-10-19 2014-10-19 00:00:00
2015-04-09 NE                24       24 2015-04-09 2015-04-09 23:00:00
           S                 24       24 2015-04-09 2015-04-09 23:00:00
           SE                24       24 2015-04-09 2015-04-09 23:00:00
2015-10-18 N                  1        1 2015-10-18 2015-10-18 00:00:00
           NE                 1        1 2015-10-18 2015-10-18 00:00:00
           S                  1        1 2015-10-18 2015-10-18 00:00:00
           SE                 1        1 2015-10-18 2015-10-18 00:00:00
2016-10-16 N                  1        1 2016-10-16 2016-10-16 00:00:00
           NE                 1        1 2016-10-16 2016-10-16 00:00:00
           S                  1        1 2016-10-16 2016-10-16 00:00:00
           SE                 1        1 2016-10-16 2016-10-16 00:00:00
2017-10-15 N                  1        1 2017-10-15 2017-10-15 00:00:00
           NE                 1        1 2017-10-15 2017-10-15 00:00:00
           S                  1        1 2017-10-15 2017-10-15 00:00:00
           SE                 1        1 2017-10-15 2017-10-15 00:00:00
2018-11-04 N                  1        1 2018-11-04 2018-11-04 00:00:00
           NE                 1        1 2018-11-04 2018-11-04 00:00:00
           SE                 1        1 2018-11-04 2018-11-04 00:00:00

#### 4.1. O que os Resultados Mostram?

Antes de analisar as datas, é importante entender a exibição da tabela. Quando as colunas min e max mostram apenas a data, sem o horário, o registro corresponde às 00:00:00. O Pandas não exibe o horário quando ele está exatamente à meia-noite.

Em 01/12/2013, cada subsistema possui 24 valores ausentes e 24 horários diferentes. O primeiro horário é 00:00 e o último é 23:00. Portanto, não existe carga válida durante todo esse dia.

Em 01/02/2014 e 09/04/2015, o mesmo acontece com Nordeste, Sul e Sudeste. O Norte não aparece na tabela de valores nulos, então ainda precisamos verificar se seus registros existem nessas duas datas.

Em 19/10/2014, 18/10/2015, 16/10/2016 e 15/10/2017, cada subsistema possui apenas um valor ausente. Como o menor e o maior horário são 00:00, a ausência ocorreu somente à meia-noite.

Em 04/11/2018, Norte, Nordeste e Sudeste também possuem uma carga ausente às 00:00. O Sul não aparece entre os nulos e será analisado separadamente.

Até aqui, conseguimos identificar quando os valores estão ausentes. No próximo tópico, vamos analisar os casos que ficaram diferentes dos demais.


### 5. Análise dos Casos Identificados

Duas situações precisam de uma verificação mais detalhada:

- o Sul não aparece entre os valores nulos de 04/11/2018 às 00:00;
- o Norte não aparece entre os valores nulos de 01/02/2014 e 09/04/2015.

#### 5.1. Subsistema Sul em 04/11/2018

Como a variável missing_data possui apenas as linhas com carga nula, vamos consultar o dataset completo. Assim, poderemos saber se o registro do Sul existe e qual valor foi armazenado nele.


In [49]:
electric_data[
    electric_data['din_instante']
    == pd.Timestamp('2018-11-04 00:00:00')
]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
660568,N,NORTE,2018-11-04,NaN
660569,NE,NORDESTE,2018-11-04,NaN
660570,S,SUL,2018-11-04,0.0
660571,SE,SUDESTE,2018-11-04,NaN


A consulta mostrou que o registro do Sul existe, mas sua carga está igual a zero. Como o Pandas não considera zero um valor nulo, essa linha não apareceu na análise anterior.

Antes de tomar uma decisão, vamos verificar se existem outros valores iguais ou menores que zero em todo o histórico.


In [50]:
electric_data[electric_data['val_cargaenergiahomwmed'] <= 0]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
660570,S,SUL,2018-11-04,0.0


Foi encontrado apenas um valor igual ou menor que zero: o registro do Sul em 04/11/2018 às 00:00.

Nesse mesmo horário, os outros três subsistemas estão com a carga ausente. Além disso, uma carga igual a zero significaria que não houve nenhuma demanda em todo o subsistema durante aquela hora, o que não combina com o comportamento dessa série.

Por isso, vamos tratar esse zero como um valor ausente, assim como aconteceu nos outros subsistemas.


In [51]:
electric_data.loc[electric_data['val_cargaenergiahomwmed'] <= 0, 'val_cargaenergiahomwmed'] = pd.NA

In [52]:
electric_data['val_cargaenergiahomwmed'].isna().sum()

np.int64(260)

Depois desse ajuste, a quantidade de cargas ausentes passou de 259 para 260.

Nenhuma linha foi removida. Apenas o valor zero do Sul foi substituído por NaN.


#### 5.2. Subsistema Norte em 01/02/2014 e 09/04/2015

O Norte não apareceu entre os valores nulos dessas duas datas. Porém, isso não quer dizer que sua carga esteja preenchida.

Como missing_data contém somente linhas existentes com carga nula, vamos consultar o dataset completo para saber se os registros do Norte realmente existem.


In [53]:
dates_to_check = [
    pd.Timestamp('2014-02-01').date(),
    pd.Timestamp('2015-04-09').date()
]

electric_data[
    (electric_data['id_subsistema'] == 'N')
    & (electric_data['din_instante'].dt.date.isin(dates_to_check))
]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed


A consulta retornou um DataFrame vazio. Isso mostra que não existem registros do Norte em 01/02/2014 e 09/04/2015.

Nos outros três subsistemas, as 24 linhas de cada dia existem, mas as cargas estão nulas. No Norte, as próprias linhas não foram registradas.

Portanto, nenhum dos quatro subsistemas possui carga válida nessas duas datas. A diferença está apenas na forma como a ausência aparece na base.

Esses dados não serão preenchidos durante a consolidação. Primeiro, precisamos definir qual frequência será utilizada nas próximas análises.


### 6. Verificação dos Registros Duplicados

Agora, vamos verificar se existem linhas repetidas no dataset.

Também será analisada a combinação entre subsistema e horário, pois cada subsistema deve possuir apenas uma carga registrada em cada instante.


In [54]:
electric_data.duplicated().sum()

np.int64(0)

In [55]:
electric_data.duplicated(
    subset=['id_subsistema', 'din_instante']
).sum()

np.int64(0)

Não foram encontradas linhas completamente duplicadas e também não existem repetições na combinação entre subsistema e horário.

Isso mostra que cada subsistema possui, no máximo, uma carga registrada por instante. Mesmo assim, ainda precisamos verificar se existem horários que não aparecem na base.


### 7. Verificação da Continuidade Temporal

O método isna identifica valores nulos em linhas existentes, mas não encontra horários que não possuem nenhuma linha registrada.

Por isso, vamos comparar a quantidade de registros entre os subsistemas e criar uma sequência horária completa para procurar outros horários ausentes.


In [56]:
electric_data['id_subsistema'].value_counts().sort_index()

id_subsistema
N     227866
NE    227914
S     227914
SE    227914
Name: count, dtype: int64

Nordeste, Sul e Sudeste possuem 227.914 registros cada. O Norte possui 227.866.

A diferença é de 48 registros, exatamente a quantidade correspondente às 24 horas de 01/02/2014 e às 24 horas de 09/04/2015.

Esse resultado está de acordo com o que encontramos anteriormente. Porém, ter a mesma quantidade de registros não garante que as outras séries estejam completas. Elas ainda podem ter horários ausentes em comum.

Para conferir isso, vamos criar uma sequência com todos os horários esperados no período.


In [57]:
expected_hours = pd.date_range(
    start=electric_data['din_instante'].min(),
    end=electric_data['din_instante'].max(),
    freq='h'
)

print(f'Quantidade esperada de horários: {len(expected_hours)}')

Quantidade esperada de horários: 227928


In [58]:
missing_hours = {}

for subsystem in sorted(electric_data['id_subsistema'].unique()):
    recorded_hours = electric_data.loc[
        electric_data['id_subsistema'] == subsystem,
        'din_instante'
    ]

    missing_hours[subsystem] = expected_hours.difference(
        recorded_hours
    )

    print(
        f'{subsystem}: '
        f'{len(missing_hours[subsystem])} horários ausentes'
    )

N: 62 horários ausentes
NE: 14 horários ausentes
S: 14 horários ausentes
SE: 14 horários ausentes


In [59]:
date_counts = []

for subsystem, hours in missing_hours.items():
    counts = pd.Series(hours).dt.date.value_counts()
    counts.name = subsystem
    date_counts.append(counts)

missing_by_date = (
    pd.concat(date_counts, axis=1)
    .fillna(0)
    .astype(int)
    .sort_index()
)

missing_by_date

,N,NE,S,SE
2000-10-08,1,1,1,1
2001-10-14,1,1,1,1
2002-11-03,1,1,1,1
2003-10-19,1,1,1,1
2004-11-02,1,1,1,1
2005-10-16,1,1,1,1
2006-11-05,1,1,1,1
2007-10-14,1,1,1,1
2008-10-19,1,1,1,1
2009-10-18,1,1,1,1


A comparação mostrou 14 datas com um horário ausente nos quatro subsistemas.

Além desses casos em comum, o Norte possui as 24 horas ausentes de 01/02/2014 e as 24 horas de 09/04/2015.

Agora, vamos visualizar quais são os 14 horários que não aparecem em nenhum subsistema.


In [60]:
common_missing_hours = missing_hours['N']

for subsystem in ['NE', 'S', 'SE']:
    common_missing_hours = common_missing_hours.intersection(
        missing_hours[subsystem]
    )

common_missing_hours

DatetimeIndex(['2000-10-08', '2001-10-14', '2002-11-03', '2003-10-19',
               '2004-11-02', '2005-10-16', '2006-11-05', '2007-10-14',
               '2008-10-19', '2009-10-18', '2010-10-17', '2011-10-16',
               '2012-10-21', '2013-10-20'],
              dtype='datetime64[ns]', freq=None)

Os 14 horários ausentes acontecem sempre às 00:00, uma vez por ano, entre 2000 e 2013.

Ao comparar essas datas com os calendários oficiais, vimos que elas correspondem ao início do horário de verão, quando os relógios eram adiantados em uma hora.

Entre 2000 e 2013, a linha das 00:00 não aparece na base. Entre 2014 e 2018, a linha passou a existir, mas sem uma carga válida. Em 2018, o Sul recebeu zero e esse valor já foi convertido para NaN.

Como o mesmo padrão aparece nos quatro subsistemas, vamos considerar esses casos como uma característica da mudança de horário. As linhas não serão criadas artificialmente.

Fontes consultadas:

- [Decreto nº 3.592/2000](https://www.planalto.gov.br/ccivil_03/decreto/d3592.htm)
- [Decreto nº 5.223/2004](https://www.planalto.gov.br/ccivil_03/_ato2004-2006/2004/decreto/d5223impressao.htm)
- [Decreto nº 6.558/2008](https://www.planalto.gov.br/ccivil_03/_ato2007-2010/2008/decreto/d6558.htm)
- [Ministério de Minas e Energia — Horário de Verão](https://www.gov.br/mme/pt-br/assuntos/secretarias/secretaria-nacional-energia-eletrica/horario-de-verao)


#### 7.1. Horários Ausentes Apenas no Norte

Depois de separar os horários ausentes em comum, vamos conferir quais aparecem somente no subsistema Norte.


In [61]:
north_only_hours = missing_hours['N'].difference(
    common_missing_hours
)

pd.Series(north_only_hours).dt.date.value_counts().sort_index()

2014-02-01    24
2015-04-09    24
Name: count, dtype: int64

Os 48 horários exclusivos do Norte correspondem exatamente aos dois dias que já foram investigados: 01/02/2014 e 09/04/2015.

Portanto, não existem outros horários ausentes apenas nesse subsistema.

Nesses dois dias, as linhas do Norte não existem. Nos outros três subsistemas, as linhas existem, mas suas cargas estão nulas. Diferente dos casos de horário de verão, aqui temos dois dias completos sem informação válida para nenhum subsistema.


### 8. Resumo dos Valores Ausentes

Depois de transformar o zero do Sul em NaN, vamos atualizar o recorte dos valores ausentes e conferir sua distribuição final por data.


In [62]:
final_missing_data = electric_data[
    electric_data['val_cargaenergiahomwmed'].isna()
].copy()

final_missing_data['data'] = (
    final_missing_data['din_instante'].dt.date
)

final_missing_data['data'].value_counts().sort_index()

data
2013-12-01    96
2014-02-01    72
2014-10-19     4
2015-04-09    72
2015-10-18     4
2016-10-16     4
2017-10-15     4
2018-11-04     4
Name: count, dtype: int64

A verificação final mostrou 260 valores de carga ausentes em linhas existentes:

- 240 estão nos três dias completos sem carga válida: 01/12/2013, 01/02/2014 e 09/04/2015;
- 20 estão nas cinco mudanças para o horário de verão entre 2014 e 2018.

Também existem 104 linhas que não aparecem na sequência horária:

- 56 correspondem às 14 mudanças para o horário de verão entre 2000 e 2013, considerando os quatro subsistemas;
- 48 correspondem aos dois dias sem registros do Norte.

Nenhum valor será preenchido e nenhuma linha será criada nesta etapa. Dessa forma, mantemos os dados como foram disponibilizados e deixamos o tratamento para o momento em que a frequência da análise for definida.


### 9. Organização e Salvamento do Dataset

Antes de salvar, vamos ordenar os registros por data e subsistema.

Depois, o dataset consolidado será armazenado na pasta data/processed. Os arquivos originais da pasta data/raw continuarão sem alterações.


In [63]:
electric_data = (
    electric_data
    .sort_values(['din_instante', 'id_subsistema'])
    .reset_index(drop=True)
)

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

output_file = (
    PROCESSED_PATH / 'curva_carga_consolidado.parquet'
)

electric_data.to_parquet(output_file, index=False)

print(f'Arquivo salvo em: {output_file}')

Arquivo salvo em: ..\data\processed\curva_carga_consolidado.parquet


### 10. Validação Final

Para finalizar, vamos carregar novamente o arquivo que foi salvo e repetir as principais verificações.

O objetivo é confirmar se a quantidade de registros, os tipos das colunas e os valores ausentes continuam corretos.


In [64]:
processed_data = pd.read_parquet(output_file)

print(
    f'Dimensão: {processed_data.shape[0]} linhas '
    f'e {processed_data.shape[1]} colunas'
)
print(
    'Valores ausentes na carga:',
    processed_data['val_cargaenergiahomwmed'].isna().sum()
)
print(
    'Linhas duplicadas:',
    processed_data.duplicated().sum()
)
print(
    'Duplicidades de subsistema e horário:',
    processed_data.duplicated(
        subset=['id_subsistema', 'din_instante']
    ).sum()
)
print(
    'Valores iguais ou menores que zero:',
    (
        processed_data['val_cargaenergiahomwmed'] <= 0
    ).sum()
)
print(
    f"Período: {processed_data['din_instante'].min()} "
    f"até {processed_data['din_instante'].max()}"
)

Dimensão: 911608 linhas e 4 colunas
Valores ausentes na carga: 260
Linhas duplicadas: 0
Duplicidades de subsistema e horário: 0
Valores iguais ou menores que zero: 0
Período: 2000-01-01 00:00:00 até 2025-12-31 23:00:00


In [65]:
processed_data.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911348 non-null  float64       
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 34.1 MB


### 11. Conclusão

O histórico de carga elétrica foi reunido em um único arquivo Parquet, com 911.608 registros entre 2000 e 2025.

Durante a verificação dos dados, encontramos os seguintes pontos:

- a coluna de carga foi convertida de object para float64;
- 259 campos vazios passaram a ser reconhecidos como valores ausentes;
- um valor igual a zero foi tratado como ausente;
- não foram encontrados registros duplicados;
- existem 260 cargas nulas em linhas registradas;
- existem 104 linhas ausentes na sequência horária;
- nenhuma carga foi preenchida e nenhuma linha foi criada.

O arquivo final foi salvo em data/processed/curva_carga_consolidado.parquet e será utilizado no próximo notebook para a análise exploratória.
